# Project 25 — Multi-Hop RAG Research Agent

## Multiple Retrieval Hops Before Answering

**Goal:** LangGraph agent decomposes complex questions into sub-questions,
retrieves evidence for each in sequence, and synthesizes a final answer.

**Stack:** LangGraph · Ollama · ChromaDB · Jupyter

### What You'll Learn
1. Question decomposition into sub-questions
2. Iterative retrieval with LangGraph state management
3. Evidence accumulation across hops
4. Single-hop vs multi-hop answer quality

### Prerequisites
- **Ollama** running with `nomic-embed-text` and `qwen3:8b`

In [ ]:
!pip install -q langchain langchain-ollama langchain-community langgraph chromadb

## Step 1 — Verify Ollama Is Running

In [ ]:
import requests
OLLAMA_BASE = "http://localhost:11434"
try:
    r = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
    r.raise_for_status()
    print(f"Ollama running — {len(r.json().get('models',[]))} model(s)")
except Exception as e:
    print(f"Cannot reach Ollama: {e}")

## Step 2 — Knowledge Base

An NLP/ML history corpus where facts are spread across documents.

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.schema import Document
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.1)
emb = OllamaEmbeddings(model="nomic-embed-text")

kb = [
    Document(page_content="The Transformer architecture was introduced by Vaswani et al. in 2017. "
        "It uses self-attention instead of recurrence for sequence processing.",
        metadata={"topic": "transformer", "id": 1}),
    Document(page_content="BERT (Bidirectional Encoder Representations from Transformers) was "
        "created by Google in 2018. It uses masked language modeling and was pre-trained "
        "on BookCorpus and English Wikipedia.",
        metadata={"topic": "bert", "id": 2}),
    Document(page_content="GPT-3 has 175 billion parameters and was trained by OpenAI in 2020. "
        "It uses autoregressive language modeling. Few-shot learning emerged as a key capability.",
        metadata={"topic": "gpt3", "id": 3}),
    Document(page_content="RAG (Retrieval-Augmented Generation) was introduced by Lewis et al. "
        "in 2020. It combines a DPR retriever with a BART generator to reduce hallucination.",
        metadata={"topic": "rag", "id": 4}),
    Document(page_content="DPR (Dense Passage Retrieval) uses dual BERT encoders for questions "
        "and passages. It outperforms BM25 on open-domain QA benchmarks.",
        metadata={"topic": "dpr", "id": 5}),
    Document(page_content="Self-attention computes query, key, value matrices from input. "
        "Attention scores = softmax(QK^T / sqrt(d_k)). Multi-head attention runs multiple "
        "parallel attention heads.",
        metadata={"topic": "attention", "id": 6}),
    Document(page_content="BART is a denoising seq2seq pre-trained model by Facebook AI (2019). "
        "It combines a bidirectional encoder (like BERT) with an autoregressive decoder (like GPT).",
        metadata={"topic": "bart", "id": 7}),
]

vs = Chroma.from_documents(kb, emb, collection_name="multihop")
ret = vs.as_retriever(search_kwargs={"k": 2})
print(f"Knowledge base: {len(kb)} documents")

## Step 3 — Single-Hop Baseline

In [ ]:
qa_p = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY the context. If info is missing, say so."),
    ("human", "Context: {context}\n\nQuestion: {question}")
])
qa_c = qa_p | llm | StrOutputParser()

complex_qs = [
    "How does the attention mechanism in BERT relate to the original Transformer?",
    "What retrieval method does RAG use, and how does it compare to keyword search?",
    "Trace the evolution from Transformers to GPT-3 — what are the key changes?",
]

print("=== SINGLE-HOP BASELINE ===\n")
for q in complex_qs:
    docs = ret.invoke(q)
    ctx = "\n".join([d.page_content for d in docs])
    ans = qa_c.invoke({"context": ctx, "question": q})
    print(f"Q: {q}")
    print(f"Sources: {[d.metadata['topic'] for d in docs]}")
    print(f"A: {str(ans)[:150]}...\n")

## Step 4 — Build Multi-Hop Graph with LangGraph

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

class ResearchState(TypedDict):
    question: str
    sub_questions: list[str]
    retrieved_context: Annotated[list[str], operator.add]
    hop_count: int
    final_answer: str

def decompose(state):
    """Break complex question into sub-questions."""
    p = ChatPromptTemplate.from_messages([
        ("system", "Break this complex question into 2-3 simpler sub-questions. "
         "Return one question per line, no numbering."),
        ("human", "{question}")
    ])
    r = (p | llm | StrOutputParser()).invoke({"question": state["question"]})
    subs = [q.strip() for q in r.strip().split("\n") if q.strip()][:3]
    print(f"  Decomposed into {len(subs)} sub-questions:")
    for i, sq in enumerate(subs): print(f"     {i+1}. {sq}")
    return {"sub_questions": subs, "hop_count": 0}

def retrieve_hop(state):
    """Retrieve documents for current sub-question."""
    hop = state["hop_count"]
    if hop < len(state["sub_questions"]):
        sub_q = state["sub_questions"][hop]
        docs = ret.invoke(sub_q)
        context = [f"[Hop {hop+1}: {sub_q}]\n" + d.page_content for d in docs]
        topics = [d.metadata["topic"] for d in docs]
        print(f"  Hop {hop+1}: {sub_q[:50]}... → {topics}")
        return {"retrieved_context": context, "hop_count": hop + 1}
    return {"hop_count": hop}

def should_continue(state):
    return "more" if state["hop_count"] < len(state["sub_questions"]) else "synth"

def synthesize(state):
    """Combine all retrieved evidence into a final answer."""
    all_ctx = "\n\n".join(state["retrieved_context"])
    p = ChatPromptTemplate.from_messages([
        ("system", "Synthesize a comprehensive answer using ALL the retrieved evidence."),
        ("human", "Question: {question}\n\nEvidence:\n{context}\n\nAnswer:")
    ])
    answer = (p | llm | StrOutputParser()).invoke(
        {"question": state["question"], "context": all_ctx})
    print(f"  Synthesized from {len(state['retrieved_context'])} evidence pieces")
    return {"final_answer": answer}

g = StateGraph(ResearchState)
g.add_node("decompose", decompose)
g.add_node("retrieve", retrieve_hop)
g.add_node("synthesize", synthesize)
g.set_entry_point("decompose")
g.add_edge("decompose", "retrieve")
g.add_conditional_edges("retrieve", should_continue,
    {"more": "retrieve", "synth": "synthesize"})
g.add_edge("synthesize", END)
app = g.compile()
print("Multi-hop research graph compiled!")

## Step 5 — Run Multi-Hop Queries

In [ ]:
print("=== MULTI-HOP RESULTS ===")
for q in complex_qs:
    print(f"\n{'='*60}\nQ: {q}\n{'-'*60}")
    result = app.invoke({
        "question": q, "sub_questions": [],
        "retrieved_context": [], "hop_count": 0, "final_answer": ""
    })
    print(f"\nFinal Answer:\n{result['final_answer']}")
    print(f"Evidence pieces: {len(result['retrieved_context'])}")

## Limitations

1. **Latency:** Each hop adds retrieval + LLM time.
2. **Decomposition quality:** Bad sub-questions lead to irrelevant hops.
3. **Error propagation:** Wrong evidence in hop 1 can mislead hop 2.
4. **Overkill for simple queries:** Single-hop is faster and often sufficient.

## What You Learned

| Concept | What We Did |
|---|---|
| **Question decomposition** | Split complex queries into sub-questions |
| **Iterative retrieval** | Retrieve evidence per sub-question |
| **LangGraph state** | Accumulate evidence across hops |
| **Conditional branching** | Continue or stop based on hop count |
| **Multi-source synthesis** | Combine evidence into final answer |